# Train

In [1]:
import sys
sys.path.append("..")
import numpy as np
from collections import Counter
from Decision_Tree.DecisionTree import DecisionTree

In [51]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []


    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(min_samples_split=self.min_samples_split, 
                                max_depth=self.max_depth, 
                                n_features=self.n_features)
            X_sample, y_sample = self._bootstrap_samples(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    # Helper function
    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]


    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees]) # return a list of lists, each inner list contains the predictions for trees in that subsample ([[tree0, tree1, tree2], [tree0, tree1, tree2], ...])
        # we want to have a list of lists, where each inner list shows the predicitons for the same tree ([[tree0 predictions], [tree1 predictions], ...])
        tree_preds = np.swapaxes(predictions, 0, 1)
        predictions = np.array([self._most_common_label(pred) for pred in tree_preds])
        return predictions

    # Helper function
    def _most_common_label(self, y):
            counter = Counter(y)
            value = counter.most_common(1)[0][0]
            return value

# Test

In [52]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np


data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

def accuracy(y_pred, y_test):
    acc = np.sum(y_pred == y_test) / len(y_test)
    return acc

In [53]:
clf = RandomForest()
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

acc = accuracy(y_test, predictions)

print(acc)

0.9122807017543859
